In [1]:
from pathlib import Path
import re
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
CSV_PATH = Path("Water Quality(2).csv")  # Change this if your CSV is in another folder
TARGET_PARAMETER = "Specific conductance"
OUTPUT_DIR = Path("AquaInsight_Task3_Outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Imports completed.")
print("Current working directory:", Path.cwd())
print("CSV path:", CSV_PATH.resolve())

Imports completed.
Current working directory: c:\Users\YASH\Desktop\BrainyBeam\daily task
CSV path: C:\Users\YASH\Desktop\BrainyBeam\daily task\Water Quality(2).csv


In [2]:
# If the CSV is not in the notebook's current folder, set CSV_PATH to its full path.
if not CSV_PATH.exists():
    alternatives = [Path("Water Quality.csv"), Path("Water Quality(2).csv"), Path("/mnt/data/Water Quality(2).csv")]
    found = next((p for p in alternatives if p.exists()), None)
    if found is None:
        raise FileNotFoundError(
            f"Could not find {CSV_PATH}. Put the CSV beside this notebook or update CSV_PATH."
        )
    CSV_PATH = found

df = pd.read_csv(CSV_PATH, low_memory=False)
required_columns = [
    "MonitoringLocationID", "ActivityStartDate",
    "CharacteristicName", "ResultValue", "ResultUnit"
]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {missing_columns}")

print(f"Loaded {len(df):,} rows and {len(df.columns)} columns.")
print("Date range in raw file:", df["ActivityStartDate"].min(), "to", df["ActivityStartDate"].max())
display(df.head())

Loaded 229,916 rows and 40 columns.
Date range in raw file: 2000-01-03 to 2023-12-06


,DatasetName,MonitoringLocationID,MonitoringLocationName,MonitoringLocationLatitude,MonitoringLocationLongitude,MonitoringLocationHorizontalCoordinateReferenceSystem,MonitoringLocationHorizontalAccuracyMeasure,MonitoringLocationHorizontalAccuracyUnit,MonitoringLocationVerticalMeasure,MonitoringLocationVerticalUnit,...,ResultStatusID,ResultComment,ResultAnalyticalMethodID,ResultAnalyticalMethodContext,ResultAnalyticalMethodName,AnalysisStartDate,AnalysisStartTime,AnalysisStartTimeZone,LaboratoryName,LaboratorySampleID
0,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,...,Preliminary,NaN,110688,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN
1,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,...,Preliminary,NaN,2011,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN
2,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,...,Preliminary,NaN,20110,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN
3,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,...,Preliminary,NaN,17209,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN
4,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,...,Preliminary,NaN,10110,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN


In [3]:
data = df.copy()
data["ActivityStartDate"] = pd.to_datetime(data["ActivityStartDate"], errors="coerce")
data["ResultValue"] = pd.to_numeric(data["ResultValue"], errors="coerce")
data["CharacteristicName"] = data["CharacteristicName"].astype("string").str.strip()
data["MonitoringLocationID"] = data["MonitoringLocationID"].astype("string").str.strip()
data["SampleDate"] = data["ActivityStartDate"].dt.normalize()

data = data.dropna(subset=[
    "MonitoringLocationID", "SampleDate", "CharacteristicName", "ResultValue"
]).copy()

# The target is selected by exact parameter name, case-insensitively.
target_mask = data["CharacteristicName"].str.casefold().eq(TARGET_PARAMETER.casefold())
target_rows = data.loc[target_mask].copy()

if target_rows.empty:
    available = sorted(data["CharacteristicName"].dropna().unique().tolist())
    raise ValueError(
        f"No rows found for target '{TARGET_PARAMETER}'. Example parameter names: {available[:30]}"
    )

target_units = target_rows["ResultUnit"].dropna().astype(str).str.strip().value_counts()
print("Cleaned data shape:", data.shape)
print("Monitoring locations:", data["MonitoringLocationID"].nunique())
print("Parameters:", data["CharacteristicName"].nunique())
print("Clean date range:", data["SampleDate"].min(), "to", data["SampleDate"].max())
print("\nTarget unit counts:")
display(target_units.to_frame("records"))

# This dataset was inspected and Specific conductance is reported as uS/cm.
# Enforce one target unit so values from different units are never mixed.
if target_units.empty:
    raise ValueError("Specific conductance has no recorded ResultUnit; verify the source data.")
if len(target_units) > 1:
    raise ValueError(
        "Specific conductance appears in multiple units. Normalize units before modeling: "
        + ", ".join(target_units.index.astype(str))
    )
TARGET_UNIT = target_units.index[0]
target_rows = target_rows[target_rows["ResultUnit"].astype(str).str.strip().eq(TARGET_UNIT)].copy()
print(f"Target: {TARGET_PARAMETER} ({TARGET_UNIT}) | target records: {len(target_rows):,}")

# To avoid combining incompatible measurement units, retain only parameters with
# exactly one non-missing reported unit, and exclude rows with unknown units.
unit_counts = (
    data.dropna(subset=["ResultUnit"])
        .assign(ResultUnit=lambda d: d["ResultUnit"].astype(str).str.strip())
        .groupby("CharacteristicName")["ResultUnit"]
        .nunique()
)
unit_consistent_parameters = set(unit_counts[unit_counts == 1].index)
if TARGET_PARAMETER not in unit_consistent_parameters:
    raise ValueError(f"Target '{TARGET_PARAMETER}' has inconsistent units and cannot be modeled safely.")

modeling_measurements = data[
    data["CharacteristicName"].isin(unit_consistent_parameters)
    & data["ResultUnit"].notna()
].copy()
modeling_measurements["ResultUnit"] = modeling_measurements["ResultUnit"].astype(str).str.strip()

print("Parameters retained after unit consistency check:", len(unit_consistent_parameters))
print("Measurement rows retained for pivot:", len(modeling_measurements))


Cleaned data shape: (229916, 41)
Monitoring locations: 58
Parameters: 83
Clean date range: 2000-01-03 00:00:00 to 2023-12-06 00:00:00

Target unit counts:


,records
ResultUnit,
uS/cm,8963


Target: Specific conductance (uS/cm) | target records: 8,963
Parameters retained after unit consistency check: 63
Measurement rows retained for pivot: 121167


In [4]:
# Median aggregation handles multiple results for the same site/date/parameter.
pivot_data = modeling_measurements.pivot_table(
    index=["MonitoringLocationID", "SampleDate"],
    columns="CharacteristicName",
    values="ResultValue",
    aggfunc="median"
).reset_index()
pivot_data.columns.name = None

# Add numeric coordinates by monitoring location, where available.
coord_columns = [
    c for c in ["MonitoringLocationLatitude", "MonitoringLocationLongitude"]
    if c in data.columns
]
if coord_columns:
    site_coordinates = (
        data.groupby("MonitoringLocationID", as_index=False)[coord_columns]
        .median(numeric_only=True)
    )
    pivot_data = pivot_data.merge(site_coordinates, on="MonitoringLocationID", how="left")

if TARGET_PARAMETER not in pivot_data.columns:
    raise ValueError(f"Target column '{TARGET_PARAMETER}' was not created by the pivot.")

# Keep rows with known target, sort chronologically, and reset row indexes.
model_data = (
    pivot_data.dropna(subset=[TARGET_PARAMETER])
    .sort_values(["SampleDate", "MonitoringLocationID"])
    .reset_index(drop=True)
)

if len(model_data) < 30:
    raise ValueError(f"Only {len(model_data)} target rows are available; not enough for this split.")

print("Pivoted table shape:", pivot_data.shape)
print("Rows with target available:", len(model_data))
print("Unique sample dates:", model_data["SampleDate"].nunique())
display(model_data.head())

Pivoted table shape: (7283, 67)
Rows with target available: 7222
Unique sample dates: 2167


,MonitoringLocationID,SampleDate,"Alkalinity, total",Ammonia,Antimony,Apparent color,Beryllium,Bismuth,Boron,Bromine,...,"Total Nitrogen, mixed forms","Total Phosphorus, mixed forms",True color,Tungsten,Uranium,Ytterbium,Yttrium,Zirconium,MonitoringLocationLatitude,MonitoringLocationLongitude
0,NS01ED0005,2000-01-03,NaN,NaN,NaN,133.0,NaN,NaN,NaN,NaN,...,0.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44.4367,-65.2228
1,NS01ED0110,2000-01-03,NaN,NaN,NaN,34.0,NaN,NaN,NaN,NaN,...,0.07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44.4236,-65.2133
2,NS01ED0111,2000-01-03,NaN,NaN,NaN,31.0,NaN,NaN,NaN,NaN,...,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44.4264,-65.2128
3,NS01EE0014,2000-01-03,NaN,NaN,NaN,119.0,NaN,NaN,NaN,NaN,...,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44.4619,-65.0483
4,NS01ED0005,2000-01-18,NaN,NaN,NaN,112.0,NaN,NaN,NaN,NaN,...,0.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44.4367,-65.2228


In [5]:
# Exclude identifiers, date, and target from predictors.
# Coordinates are retained if available; other parameters are candidate predictors.
# "Conductivity" is excluded because it can be nearly synonymous with the target.
excluded_columns = {"MonitoringLocationID", "SampleDate", TARGET_PARAMETER, "Conductivity"}
candidate_features = [
    c for c in model_data.columns
    if c not in excluded_columns
]

# Ensure predictors are numeric and exclude columns that are completely missing.
X = model_data[candidate_features].copy()
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.loc[:, X.notna().any(axis=0)]

y = pd.to_numeric(model_data[TARGET_PARAMETER], errors="coerce")
valid_target = y.notna()
X = X.loc[valid_target].reset_index(drop=True)
y = y.loc[valid_target].reset_index(drop=True)
model_data = model_data.loc[valid_target].reset_index(drop=True)

if X.shape[1] == 0:
    raise ValueError("No usable numeric predictors were found after preprocessing.")

# LightGBM and some libraries are strict about punctuation in feature names.
# Sanitize names and make them unique; values remain unchanged.
def clean_feature_name(name):
    cleaned = re.sub(r"[^A-Za-z0-9_]+", "_", str(name))
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")
    return cleaned or "feature"

new_names = []
name_counts = {}
for original in X.columns:
    base = clean_feature_name(original)
    count = name_counts.get(base, 0)
    name_counts[base] = count + 1
    new_names.append(base if count == 0 else f"{base}_{count}")
X.columns = new_names

sample_dates = model_data["SampleDate"].copy()

print("X (features) shape:", X.shape)
print("y (target) shape:", y.shape)
print("Target:", TARGET_PARAMETER, f"({TARGET_UNIT})")
print("Feature columns:", X.columns.tolist())
display(X.head())
display(y.head())

X (features) shape: (7222, 63)
y (target) shape: (7222,)
Target: Specific conductance (uS/cm)
Feature columns: ['Alkalinity_total', 'Ammonia', 'Antimony', 'Apparent_color', 'Beryllium', 'Bismuth', 'Boron', 'Bromine', 'Calcium', 'Cerium', 'Cesium', 'Chloride', 'Chlorophyll_a_uncorrected_for_pheophytin', 'Depth_of_water_column', 'Dissolved_oxygen_DO', 'Dissolved_oxygen_saturation', 'Europium', 'Fecal_Coliform', 'Fluoride', 'Gadolinium', 'Gallium', 'Germanium', 'Gran_acid_neutralizing_capacity', 'Hafnium', 'Hardness_carbonate', 'Holmium', 'Inorganic_carbon', 'Inorganic_nitrogen_nitrate_and_nitrite', 'Lanthanum', 'Lutetium', 'Magnesium', 'Mercury', 'Neodymium', 'Niobium', 'Nitrate', 'Nitrite', 'Organic_carbon', 'Orthophosphate', 'Potassium', 'Praseodymium', 'Rubidium', 'Samarium', 'Scandium', 'Silicate', 'Sodium', 'Sulfate', 'Suspended_Sediment_Concentration_SSC', 'Temperature_air', 'Temperature_water', 'Terbium', 'Thallium', 'Tin', 'Titanium', 'Total_Nitrogen_mixed_forms', 'Total_Phosphor

,Alkalinity_total,Ammonia,Antimony,Apparent_color,Beryllium,Bismuth,Boron,Bromine,Calcium,Cerium,...,Total_Nitrogen_mixed_forms,Total_Phosphorus_mixed_forms,True_color,Tungsten,Uranium,Ytterbium,Yttrium,Zirconium,MonitoringLocationLatitude,MonitoringLocationLongitude
0,NaN,NaN,NaN,133.0,NaN,NaN,NaN,NaN,1.10,NaN,...,0.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44.4367,-65.2228
1,NaN,NaN,NaN,34.0,NaN,NaN,NaN,NaN,0.70,NaN,...,0.07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44.4236,-65.2133
2,NaN,NaN,NaN,31.0,NaN,NaN,NaN,NaN,0.55,NaN,...,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44.4264,-65.2128
3,NaN,NaN,NaN,119.0,NaN,NaN,NaN,NaN,0.78,NaN,...,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44.4619,-65.0483
4,NaN,NaN,NaN,112.0,NaN,NaN,NaN,NaN,1.01,NaN,...,0.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44.4367,-65.2228


0    39.3
1    29.3
2    27.0
3    31.6
4    37.8
Name: Specific conductance, dtype: float64

In [6]:
# Split by unique dates so records collected on the same date cannot cross the holdout boundary.
unique_dates = np.array(sorted(sample_dates.dropna().unique()))
if len(unique_dates) < 5:
    raise ValueError("Too few unique sample dates for a chronological holdout split.")

cutoff_position = int(np.floor(len(unique_dates) * 0.80))
cutoff_position = min(max(cutoff_position, 1), len(unique_dates) - 1)
cutoff_date = pd.Timestamp(unique_dates[cutoff_position])

train_mask = sample_dates < cutoff_date
test_mask = sample_dates >= cutoff_date

X_train = X.loc[train_mask].reset_index(drop=True)
X_test = X.loc[test_mask].reset_index(drop=True)
y_train = y.loc[train_mask].reset_index(drop=True)
y_test = y.loc[test_mask].reset_index(drop=True)
dates_train = sample_dates.loc[train_mask].reset_index(drop=True)
dates_test = sample_dates.loc[test_mask].reset_index(drop=True)

if len(X_train) == 0 or len(X_test) == 0:
    raise ValueError("Chronological split produced an empty training or testing set.")

print("Cutoff date:", cutoff_date.date())
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test: ", X_test.shape, "| y_test: ", y_test.shape)
print("Training date range:", dates_train.min().date(), "to", dates_train.max().date())
print("Testing date range: ", dates_test.min().date(), "to", dates_test.max().date())
print("Date overlap:", len(set(dates_train).intersection(set(dates_test))))

Cutoff date: 2017-11-02
X_train: (5838, 63) | y_train: (5838,)
X_test:  (1384, 63) | y_test:  (1384,)
Training date range: 2000-01-03 to 2017-11-01
Testing date range:  2017-11-02 to 2023-12-06
Date overlap: 0


In [7]:
# Build folds using unique dates, then map each fold back to row positions.
# This prevents rows from the same sample date appearing in both a fold's train and validation set.
def date_aware_splits(dates, n_splits=3):
    dates = pd.to_datetime(pd.Series(dates)).reset_index(drop=True)
    unique = np.array(sorted(dates.dropna().unique()))
    if len(unique) < n_splits + 1:
        raise ValueError(
            f"Need at least {n_splits + 1} unique training dates for {n_splits}-fold CV; found {len(unique)}."
        )
    splitter = TimeSeriesSplit(n_splits=n_splits)
    splits = []
    for train_date_idx, val_date_idx in splitter.split(unique):
        train_dates_fold = set(unique[train_date_idx])
        val_dates_fold = set(unique[val_date_idx])
        tr = np.flatnonzero(dates.isin(train_dates_fold).to_numpy())
        va = np.flatnonzero(dates.isin(val_dates_fold).to_numpy())
        if len(tr) and len(va):
            splits.append((tr, va))
    if not splits:
        raise ValueError("Could not create date-aware cross-validation folds.")
    return splits

cv_splits = date_aware_splits(dates_train, n_splits=3)
print("Date-aware CV folds:", len(cv_splits))
for i, (tr, va) in enumerate(cv_splits, start=1):
    print(
        f"Fold {i}: train rows={len(tr):,}, validation rows={len(va):,}, "
        f"train max date={dates_train.iloc[tr].max().date()}, "
        f"validation min date={dates_train.iloc[va].min().date()}"
    )

Date-aware CV folds: 3
Fold 1: train rows=1,490, validation rows=1,491, train max date=2005-02-13, validation min date=2005-02-21
Fold 2: train rows=2,981, validation rows=1,435, train max date=2009-03-01, validation min date=2009-03-03
Fold 3: train rows=4,416, validation rows=1,422, train max date=2013-04-10, validation min date=2013-04-16


In [9]:
# Baseline predicts the median target value from training data.
baseline = DummyRegressor(strategy="median")
baseline.fit(np.zeros((len(y_train), 1)), y_train)
baseline_predictions = baseline.predict(np.zeros((len(y_test), 1)))

# Imputation is fitted on training data only, through a pipeline.
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
    ("model", RandomForestRegressor(
        n_estimators=250,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])
rf_pipeline.fit(X_train, y_train)
rf_predictions = rf_pipeline.predict(X_test)

extra_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
    ("model", ExtraTreesRegressor(
        n_estimators=250,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])
extra_pipeline.fit(X_train, y_train)
extra_predictions = extra_pipeline.predict(X_test)

predictions = {
    "Median baseline": baseline_predictions,
    "Random Forest": rf_predictions,
    "Extra Trees": extra_predictions
}
trained_models = {
    "Random Forest": rf_pipeline,
    "Extra Trees": extra_pipeline
}

print("Baseline, Random Forest, and Extra Trees trained.")

Baseline, Random Forest, and Extra Trees trained.


In [10]:
# Optional packages are handled gracefully. If not installed, this cell skips them
# instead of stopping the whole notebook.
xgb_search = None
lgb_search = None

try:
    from xgboost import XGBRegressor
    xgb_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("model", XGBRegressor(
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist"
        ))
    ])
    xgb_params = {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0]
    }
    xgb_search = RandomizedSearchCV(
        xgb_pipeline,
        param_distributions=xgb_params,
        n_iter=6,
        scoring="neg_root_mean_squared_error",
        cv=cv_splits,
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbose=1,
        error_score="raise"
    )
    print("Tuning XGBoost...")
    xgb_search.fit(X_train, y_train)
    predictions["XGBoost"] = xgb_search.best_estimator_.predict(X_test)
    trained_models["XGBoost"] = xgb_search.best_estimator_
    print("XGBoost best parameters:", xgb_search.best_params_)
except ImportError:
    print("XGBoost is not installed; skipping it. The other models will still run.")
except Exception as exc:
    print("XGBoost tuning failed and was skipped:", repr(exc))

try:
    from lightgbm import LGBMRegressor
    lgb_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("model", LGBMRegressor(
            random_state=RANDOM_STATE,
            verbosity=-1,
            n_jobs=-1
        ))
    ])
    lgb_params = {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [-1, 5, 10],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__num_leaves": [15, 31, 50],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0]
    }
    lgb_search = RandomizedSearchCV(
        lgb_pipeline,
        param_distributions=lgb_params,
        n_iter=6,
        scoring="neg_root_mean_squared_error",
        cv=cv_splits,
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbose=1,
        error_score="raise"
    )
    print("\nTuning LightGBM...")
    lgb_search.fit(X_train, y_train)
    predictions["LightGBM"] = lgb_search.best_estimator_.predict(X_test)
    trained_models["LightGBM"] = lgb_search.best_estimator_
    print("LightGBM best parameters:", lgb_search.best_params_)
except ImportError:
    print("LightGBM is not installed; skipping it. The other models will still run.")
except Exception as exc:
    print("LightGBM tuning failed and was skipped:", repr(exc))

print("\nAvailable prediction models:", list(predictions.keys()))

Tuning XGBoost...
Fitting 3 folds for each of 6 candidates, totalling 18 fits
XGBoost best parameters: {'model__subsample': 0.8, 'model__n_estimators': 300, 'model__max_depth': 3, 'model__learning_rate': 0.03, 'model__colsample_bytree': 0.8}

Tuning LightGBM...
Fitting 3 folds for each of 6 candidates, totalling 18 fits
LightGBM best parameters: {'model__subsample': 0.8, 'model__num_leaves': 15, 'model__n_estimators': 200, 'model__max_depth': 5, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.8}

Available prediction models: ['Median baseline', 'Random Forest', 'Extra Trees', 'XGBoost', 'LightGBM']


In [11]:
def metric_row(name, actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return {
        "Model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "R2": r2_score(actual, predicted) if len(actual) > 1 else np.nan
    }

test_metrics = pd.DataFrame([
    metric_row(name, y_test, pred)
    for name, pred in predictions.items()
]).sort_values("RMSE").reset_index(drop=True)

display(test_metrics.round(4))
metrics_path = OUTPUT_DIR / "AquaInsight_Task3_Holdout_Metrics.csv"
test_metrics.to_csv(metrics_path, index=False)
print("Saved:", metrics_path.resolve())

,Model,MAE,RMSE,R2
0,Extra Trees,8.2629,39.8295,0.9393
1,Random Forest,8.9705,40.4324,0.9375
2,LightGBM,12.7118,44.4792,0.9243
3,XGBoost,11.5868,45.1146,0.9222
4,Median baseline,73.5168,175.6932,-0.1807


Saved: C:\Users\YASH\Desktop\BrainyBeam\daily task\AquaInsight_Task3_Outputs\AquaInsight_Task3_Holdout_Metrics.csv
